# 面试题：工具后置条件怎样设计与验证？

回答要点：后置条件是写动作结束后可由权威系统验证的业务谓词，不是模型对成功文本的解释。调拨必须满足源仓扣减、目标仓预留、单据状态与版本/金额一致；不满足时应为 failed 或 pending，并触发回读、补偿、重规划或人工升级。下面比较 HTTP 200 与显式后置条件。

## 真实案例

仓储 Agent 创建调拨单，六个事件包含接口 200 但源仓未扣减、目标仓未预留、最终一致延迟与完整成功。

## 基线

基线把 200 响应作为完成。

## 结果解读

手写验证器输出三个后置谓词与完成状态。

## 失败案例

部分写入返回 200 时，不能继续通知仓库已完成调拨。

In [1]:
events = [{'id':'P1','http':200,'source_delta':-5,'target_reserved':5,'status':'created','version':2}, {'id':'P2','http':200,'source_delta':0,'target_reserved':5,'status':'created','version':2}, {'id':'P3','http':200,'source_delta':-5,'target_reserved':0,'status':'created','version':2}, {'id':'P4','http':202,'source_delta':-5,'target_reserved':5,'status':'pending','version':1}, {'id':'P5','http':500,'source_delta':0,'target_reserved':0,'status':'failed','version':1}, {'id':'P6','http':200,'source_delta':-3,'target_reserved':3,'status':'created','version':4}]  # 构造六条调拨工具响应和权威仓储状态。
print('调拨事件:', events)  # 输出接口响应与读回的领域字段。
print('教学说明：源仓变化和目标预留由权威库存系统返回，不由模型猜测。')  # 强调后置验证的数据来源。

调拨事件: [{'id': 'P1', 'http': 200, 'source_delta': -5, 'target_reserved': 5, 'status': 'created', 'version': 2}, {'id': 'P2', 'http': 200, 'source_delta': 0, 'target_reserved': 5, 'status': 'created', 'version': 2}, {'id': 'P3', 'http': 200, 'source_delta': -5, 'target_reserved': 0, 'status': 'created', 'version': 2}, {'id': 'P4', 'http': 202, 'source_delta': -5, 'target_reserved': 5, 'status': 'pending', 'version': 1}, {'id': 'P5', 'http': 500, 'source_delta': 0, 'target_reserved': 0, 'status': 'failed', 'version': 1}, {'id': 'P6', 'http': 200, 'source_delta': -3, 'target_reserved': 3, 'status': 'created', 'version': 4}]
教学说明：源仓变化和目标预留由权威库存系统返回，不由模型猜测。


In [2]:
baseline = [(row['id'], '完成' if row['http'] == 200 else '失败') for row in events]  # 构造只根据 HTTP 状态码判断的基线。
print('HTTP 基线:', baseline)  # 输出 P2/P3 被错误宣布完成的结果。
print('基线问题：传输成功不等于跨仓领域状态完整。')  # 解释 2xx 的语义边界。

HTTP 基线: [('P1', '完成'), ('P2', '完成'), ('P3', '完成'), ('P4', '失败'), ('P5', '失败'), ('P6', '完成')]
基线问题：传输成功不等于跨仓领域状态完整。


In [3]:
def verify_transfer(row):  # 定义创建调拨后的领域后置条件验证器。
    quantity = abs(row['source_delta'])  # 从源仓变化推导本次调拨数量。
    source_ok = row['source_delta'] < 0  # 检查源仓库存是否真的扣减。
    target_ok = row['target_reserved'] == quantity  # 检查目标仓预留是否与数量一致。
    status_ok = row['status'] == 'created'  # 检查调拨单是否进入可继续的创建状态。
    if source_ok and target_ok and status_ok:  # 只有三项后置条件都成立才确认完成。
        return 'verified', {'source':source_ok,'target':target_ok,'status':status_ok}  # 返回可审计成功证据。
    if row['status'] == 'pending':  # 最终一致系统的中间状态不能伪装失败或成功。
        return 'pending_readback', {'source':source_ok,'target':target_ok,'status':status_ok}  # 请求在一致性窗口后再回读。
    return 'failed_or_partial', {'source':source_ok,'target':target_ok,'status':status_ok}  # 返回部分写入或失败结论。

In [4]:
results = [(row['id'],) + verify_transfer(row) for row in events]  # 对六条调拨事件执行后置条件验证。
print('id | 验证结论 | 后置条件证据')  # 输出结果表标题。
for item in results:  # 遍历每个事件的状态与三个谓词。
    print(item[0], item[1], item[2])  # 输出权威后置检查结果。
print('HTTP 误报数:', sum(dict(baseline)[row['id']] == '完成' and verify_transfer(row)[0] != 'verified' for row in events))  # 统计只看 HTTP 的错误完成。

id | 验证结论 | 后置条件证据
P1 verified {'source': True, 'target': True, 'status': True}
P2 failed_or_partial {'source': False, 'target': False, 'status': True}
P3 failed_or_partial {'source': True, 'target': False, 'status': True}
P4 pending_readback {'source': True, 'target': True, 'status': False}
P5 failed_or_partial {'source': False, 'target': True, 'status': False}
P6 verified {'source': True, 'target': True, 'status': True}
HTTP 误报数: 2


In [5]:
wrong = dict(baseline)['P2']  # 读取源仓未扣减时 HTTP 基线的错误完成。
fixed = dict((item[0], item[1]) for item in results)['P2']  # 读取同一事件的后置验证结论。
print('失败案例 P2：HTTP 基线=', wrong, '，后置验证=', fixed)  # 展示 200 也可能是部分写入。
print('生产差距：需定义一致性窗口、关联 ID、轮询/回调、补偿动作和每种业务操作的可验证后置契约。')  # 说明生产设计。

失败案例 P2：HTTP 基线= 完成 ，后置验证= failed_or_partial
生产差距：需定义一致性窗口、关联 ID、轮询/回调、补偿动作和每种业务操作的可验证后置契约。


In [6]:
assert verify_transfer(events[0])[0] == 'verified'  # 验证完整源仓、目标仓与单据状态可确认完成。
assert verify_transfer(events[1])[0] == 'failed_or_partial'  # 验证源仓未扣减的部分写入不能完成。
assert verify_transfer(events[3])[0] == 'pending_readback'  # 验证 202 pending 进入回读而非伪造结论。